In [204]:
import pandas as pd
import re, math
import nltk
from collections import Counter
from nltk.stem import PorterStemmer

In [205]:
# 1) Load dataset
csv_path = "Fin_lab-PRProject_dataset.csv"
df = pd.read_csv(csv_path)

In [206]:
# pick the 'review' column as text data
text_col = 'review'
docs_raw = df[text_col].astype(str).tolist()

In [207]:
# 2) NLTK stopwords + PorterStemmer
try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except LookupError:
    print("NLTK stopwords not found. Downloading...")
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))

In [208]:
stemmer = PorterStemmer()

In [209]:
# Extended stopwords with a few extreme common tokens in reviews
extra = {'rt','amp'}
stop_words |= extra

In [210]:
# 3) Preprocessing function
def preprocess(text):
    # remove non-alphanumeric (keep spaces), lowercase, tokenise
    text = str(text)
    text = re.sub(r'[^A-Za-z0-9\s]', ' ', text)
    tokens = text.split()
    tokens = [t.lower() for t in tokens]
    # remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # apply porter stemmer
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

In [211]:
# 4) If dataset is large, find a documents that contain any of the target words
targets = ['business','making','support','data','system']
pattern = r'\b(' + '|'.join(targets) + r')\b'

In [212]:
# Use a fast vectorized filter if dataset large; otherwise fallback to scanning
mask = df[text_col].astype(str).str.contains(pattern, case=False, regex=True, na=False)
matched_indices = list(mask[mask].index)

C:\Users\Jestoni Andales\AppData\Local\Temp\ipykernel_7488\1715355963.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[text_col].astype(str).str.contains(pattern, case=False, regex=True, na=False)


In [213]:
# For demonstration / controlled manual computation, limit to first K matched documents
K = 100
if len(matched_indices) == 0:
    # If no matches, just take first K docs
    corpus_indices = list(range(min(K, len(docs_raw))))
else:
    corpus_indices = matched_indices[:K]

In [214]:
corpus = [docs_raw[i] for i in corpus_indices]

In [215]:
# 5) Preprocess each document
pre_docs = [preprocess(d) for d in corpus]

In [226]:
pre_docs

[['love',
  'game',
  'much',
  'ive',
  'brought',
  'friend',
  'famili',
  'alik',
  'wont',
  'stop',
  'plug',
  'ever',
  'seen',
  'someth',
  'cute',
  'grotesqu',
  'time',
  'caus',
  'caus',
  'game',
  'reason',
  'buy',
  '1',
  'virtual',
  'infinit',
  'replay',
  'even',
  'without',
  'mod',
  'btw',
  'tonn',
  'http',
  'www',
  'moddingofisaac',
  'com',
  '2',
  'manag',
  'casual',
  'pro',
  'time',
  'take',
  'great',
  'skill',
  'good',
  'time',
  'slowli',
  'get',
  'better',
  'play',
  'coffe',
  'break',
  'though',
  'play',
  'alot',
  'caus',
  'much',
  'fun',
  '3',
  'funni',
  '4',
  'risk',
  'vs',
  'reward',
  'game',
  'make',
  'heart',
  'race',
  '5',
  'mysteri',
  'discoveri',
  'game',
  'give',
  'hint',
  'explain',
  'anyth',
  'control',
  'hold',
  'hand',
  '6',
  'point',
  'also',
  'said',
  'stori',
  'amaz',
  'symbol',
  'imageri',
  'underton',
  'subtext',
  'left',
  'interpret',
  'buy',
  'game',
  'would',
  'recommend

In [216]:
# 6) Build vocabulary and BOW matrix
vocab = sorted(set(term for doc in pre_docs for term in doc))
bow = [Counter(doc) for doc in pre_docs]

In [217]:
# Convert to DataFrame for nicer display (terms as columns)
bow_df = pd.DataFrame(bow).fillna(0).astype(int)
bow_df = bow_df.reindex(sorted(bow_df.columns), axis=1)

In [218]:
# 7) TF: raw counts and normalized TF (term_count / total_terms_in_doc)
tf_raw = bow_df.copy()
tf_norm = tf_raw.astype(float)
for i, cnts in enumerate(bow):
    total = sum(cnts.values()) or 1
    tf_norm.iloc[i] = tf_norm.iloc[i] / total

In [219]:
# 8) DF and IDF (manual)
N = len(pre_docs)
df_counts = (bow_df > 0).sum(axis=0)

In [220]:
idf = df_counts.apply(lambda d: math.log(N / d) if d > 0 else 0.0)

In [221]:
# 9) TF-IDF manual: TF_norm * IDF
tfidf = tf_norm * idf

In [222]:
# 10) bow_df outputs
bow_df

,0,00,000,1,10,100,1000,10x,11,111,...,yea,year,yell,yet,yh,youl,young,zelda,zero,zone
0,0,0,0,1,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,0,0,0,0,7,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
98,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [223]:
# 10) tfidf outputs
tfidf.round(100)

,0,00,000,1,10,100,1000,10x,11,111,...,yea,year,yell,yet,yh,youl,young,zelda,zero,zone
0,0.0,0.0,0.0,0.015806,0.000000,0.01494,0.0,0.0,0.000000,0.0,...,0.0,0.010658,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.013471,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.008921,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.004331,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.011275,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
96,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
97,0.0,0.0,0.0,0.000000,0.132810,0.00000,0.0,0.0,0.055099,0.0,...,0.0,0.000000,0.0,0.035574,0.0,0.0,0.0,0.000000,0.0,0.0
98,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0


In [224]:
# 11) Present step-by-step for the specific target words
targets_stemmed = {t: stemmer.stem(t) for t in targets}
print("Using stemmed forms:", targets_stemmed)

Using stemmed forms: {'business': 'busi', 'making': 'make', 'support': 'support', 'data': 'data', 'system': 'system'}


In [225]:
for term_original, term_stem in targets_stemmed.items():
    df_term = int(df_counts.get(term_stem, 0)) if term_stem in df_counts.index else 0
    idf_term = float(idf.get(term_stem, 0.0)) if term_stem in idf.index else 0.0
    print("\n=== Term:", term_original, "| stemmed:", term_stem, "===")
    print(" Document Frequency (df):", df_term)
    print(" IDF = ln(N/df):", round(idf_term, 8))
    # per-document TF and TF-IDF
    for doc_idx in range(N):
        tf_raw_val = int(tf_raw.iloc[doc_idx].get(term_stem, 0)) if term_stem in tf_raw.columns else 0
        tf_norm_val = float(tf_norm.iloc[doc_idx].get(term_stem, 0.0)) if term_stem in tf_norm.columns else 0.0
        tfidf_val = float(tfidf.iloc[doc_idx].get(term_stem, 0.0)) if term_stem in tfidf.columns else 0.0
        print(f" Doc original index {corpus_indices[doc_idx]}: TF raw = {tf_raw_val}, TF norm = {tf_norm_val:.6f}, TF-IDF = {tfidf_val:.8f}")



=== Term: business | stemmed: busi ===
 Document Frequency (df): 1
 IDF = ln(N/df): 4.60517019
 Doc original index 73: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 267: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 270: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 327: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 473: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 511: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 740: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 821: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 848: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 1045: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 1259: TF raw = 0, TF norm = 0.000000, TF-IDF = 0.00000000
 Doc original index 1273: TF raw = 0, TF norm = 0.000000